<a href="https://colab.research.google.com/github/Sriyansh-36-AI-NITJ/Deep-Learning-Lab/blob/main/MLP_Image_classofication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Create the 'dataset' directory if it doesn't exist
if not os.path.exists('dataset'):
    os.makedirs('dataset')
    print("Created 'dataset' directory. Please upload your room images here.")

# Check if directory is empty to avoid further errors
dataset_path = os.listdir('dataset')
if not dataset_path:
    print("Warning: The 'dataset' directory is empty. Please add subfolders for each room type.")

room_types = os.listdir('dataset')
print("Room types found:", room_types)

print("Total categories found: ", len(dataset_path))

Created 'dataset' directory. Please upload your room images here.
Room types found: []
Total categories found:  0


In [3]:
rooms = []

for item in room_types:
 # Get all the file names
 all_rooms = os.listdir('dataset' + '/' +item)
 #print(all_rooms)

 # Add them to the list
 for room in all_rooms:
    rooms.append((item, str('dataset' + '/' +item) + '/' + room))
    print(rooms[:1])


In [4]:
# Build a dataframe
rooms_df = pd.DataFrame(data=rooms, columns=['room type', 'image'])
print(rooms_df.head())
print(rooms_df.tail())

Empty DataFrame
Columns: [room type, image]
Index: []
Empty DataFrame
Columns: [room type, image]
Index: []


In [5]:
# Let's check how many samples for each category are present
print("Total number of rooms in the dataset: ", len(rooms_df))

Total number of rooms in the dataset:  0


In [6]:
room_count = rooms_df['room type'].value_counts()

print("rooms in each category: ")
print(room_count)

rooms in each category: 
Series([], Name: count, dtype: int64)


In [7]:
import cv2
path = 'dataset/'


im_size = 100

images = []
labels = []

for i in room_types:
    data_path = path + str(i)  # entered in 1st folder and then 2nd folder and then 3rd folder
    #filenames = [i for i in os.listdir(data_path) if i.endswith('.jpg')]
    filenames = [i for i in os.listdir(data_path) ]
   # print(filenames)  # will get the names of all images which ends with .jpg extension
    for f in filenames:
        img = cv2.imread(data_path + '/' + f)  # reading that image as array
        #print(img)  # will get the image as an array
        img = cv2.resize(img, (im_size, im_size))
        images.append(img)
        labels.append(i)

In [8]:
# Transform the image array to a numpy type

images = np.array(images)

images.shape

(0,)

In [9]:
images = images.astype('float32') / 255.0

In [10]:
images.shape

(0,)

In [11]:
from sklearn.preprocessing import LabelEncoder , OneHotEncoder


y=rooms_df['room type'].values
print(y[:5])

[]


In [12]:
# for y
y_labelencoder = LabelEncoder ()
y = y_labelencoder.fit_transform (y)
print (y)

[]


In [14]:
y = y.reshape(-1, 1)

# Modern OneHotEncoder does not use 'categorical_features'
# handle_unknown='ignore' is safer for empty or inconsistent datasets
onehotencoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

if len(y) > 0:
    Y = onehotencoder.fit_transform(y)
    print("Shape of Y:", Y.shape)
else:
    Y = np.array([])
    print("Warning: 'y' is empty. Please upload data to the 'dataset' folder and re-run previous cells.")

In [22]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

if len(images) > 0 and len(Y) > 0:
    images, Y = shuffle(images, Y, random_state=1)
    train_x, test_x, train_y, test_y = train_test_split(images, Y, test_size=0.05, random_state=415)

    print("Training set shape:", train_x.shape)
    print("Testing set shape:", test_x.shape)
else:
    print("Error: Dataset is empty. Please upload images to subfolders within the 'dataset' directory.")

Error: Dataset is empty. Please upload images to subfolders within the 'dataset' directory.


In [27]:
if 'train_x' in locals() and len(train_x) > 0:
    # Using dynamic shape instead of hardcoded (373, 30000)
    train_x = np.reshape(train_x, (train_x.shape[0], n_input))
    print("Train shape:", train_x.shape)

    test_x = np.reshape(test_x, (test_x.shape[0], n_input))
    print("Test shape:", test_x.shape)
else:
    print("Waiting for data... Please upload images and run previous cells first.")

Waiting for data... Please upload images and run previous cells first.


In [ ]:
# hyper Parameters
learning_rate = 0.001


In [ ]:
# Network Parameters
n_hidden_1 = 256 # 1st layer has 256 neurons
n_hidden_2 = 256 # 2nd layer has 256 neurons
n_input = 30000 #  data input (img shape: 100*100*3)
n_classes = 3 # total classes (3)


In [17]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

# Network Parameters - moved here to ensure they are defined before use
n_hidden_1 = 256
n_hidden_2 = 256
n_input = 30000
n_classes = 3

# tf Graph input
x = tf.placeholder("float", [None, n_input])
y_ = tf.placeholder("float", [None, n_classes])

In [18]:
# Store layers weight & bias
weights = {
    'h1': tf.Variable(tf.random_normal([n_input, n_hidden_1])),  # n_input 30000 and n_hidden_1 256
    'h2': tf.Variable(tf.random_normal([n_hidden_1, n_hidden_2])),  # output of h1 will be a input for h2 and n_hidden_2 256
    'out': tf.Variable(tf.random_normal([n_hidden_2, n_classes]))  # 256,3
}
biases = {
    'b1': tf.Variable(tf.random_normal([n_hidden_1])),
    'b2': tf.Variable(tf.random_normal([n_hidden_2])),
    'out': tf.Variable(tf.random_normal([n_classes]))
}

In [19]:
# Create model
def multilayer_perceptron(x, weights, biases):
    # Hidden layer with RELU activation
    layer_1 = tf.add(tf.matmul(x, weights['h1']), biases['b1'])  #matmul on x and w , then adding it with bias # summation
    layer_1 = tf.nn.relu(layer_1)   # activation function

    # Hidden layer with RELU activation
    layer_2 = tf.add(tf.matmul(layer_1, weights['h2']), biases['b2'])
    layer_2 = tf.nn.relu(layer_2)

    # Output layer with linear activation
    out_layer = tf.matmul(layer_2, weights['out']) + biases['out']
    out_layer = tf.nn.softmax(out_layer)
    return out_layer


In [20]:
# Construct model
pred = multilayer_perceptron(x, weights, biases)


In [23]:
# Define hyperparameters
learning_rate = 0.001

# Define loss and optimizer
# pred is the output of MLP. y_ is the placeholder for target output
cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=pred, labels=y_))

# Adam Optimizer
optimizer = tf.train.AdamOptimizer(learning_rate=learning_rate).minimize(cost)

In [24]:
# Initializing the variables
init = tf.global_variables_initializer()
#create an empty list to store the cost history and accuracy history
cost_history = []
accuracy_history = []
# Launch the graph


In [25]:
# the execution
sess = tf.Session()
sess.run(init)

In [26]:
cost_history = []
n_epochs = 20

#ValueError: setting an array element with a sequence will resolve by 2 below given lines
train_y=train_y.todense()
#print(train_y)


for i in range(n_epochs):
    a, c = sess.run([optimizer, cost], feed_dict={x: train_x, y_: train_y})  #working
    cost_history = np.append(cost_history,c)  # working
    print('epoch : ', i,  ' - ', 'cost: ', c) #working


NameError: name 'train_y' is not defined

# checkig accuracy of model on test data


In [ ]:


test_y=test_y.todense()  #working solution of ValueError: setting an array element with a sequence.
print(test_y)


# toarray returns an ndarray; todense returns a matrix. If you want a matrix, use todense; otherwise, use toarray.


correct_prediction = tf.equal(tf.argmax(pred,1), tf.argmax(y_,1))
correct_prediction

accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
accuracy


# retrun the accuracy on the test set.
print("Accuracy: ", sess.run(accuracy, feed_dict={x: test_x, y_:test_y}))

# take input from user and predict the result

In [ ]:

import cv2
img = cv2.imread('room_test_img.jpg')

img = cv2.resize(img, (100, 100))
#nx, ny, nc = 100,100,3
#img_flat = img.reshape(nx * ny * nc)
IMG = np.reshape(img,(1,30000))
pred_y = sess.run(pred, feed_dict={x: IMG})
print(pred_y)